# 🌱 CNN Image Classification — Beans Disease Dataset

**Dataset:** [AI-Lab-Makerere/beans](https://huggingface.co/datasets/AI-Lab-Makerere/beans)

**Kelas (3 kelas):** `angular_leaf_spot` · `bean_rust` · `healthy`

| # | Kriteria | Status |
|---|----------|--------|
| 1 | Dataset bebas, min. 1000 gambar | ✅ ~1.295 gambar asli |
| 2 | Bukan dataset RPS / X-Ray | ✅ Beans Disease |
| 3 | Split Train / Validation / Test | ✅ 70 / 15 / 15 |
| 4 | Sequential + Conv2D + Pooling | ✅ |
| 5 | Akurasi train & test ≥ 85% | ✅ Target |
| 6 | Plot Akurasi & Loss | ✅ |
| 7 | SavedModel + TF-Lite + TFJS | ✅ |
| + | Callback (EarlyStopping, ReduceLR, Checkpoint) | ✅ |
| + | Resolusi gambar tidak seragam | ✅ |
| + | Min. 3 kelas | ✅ 3 kelas |
| + | Inference TF-Lite + bukti output | ✅ |

## 0. Setup & Install Dependencies

In [ ]:
!pip install -q "packaging>=24.2.0"
!pip install -q datasets tensorflowjs Pillow

import os, math, random, shutil, warnings, subprocess, zipfile
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

## 1. Download Dataset via Hugging Face

> Dataset **Beans Disease** diunduh dari `AI-Lab-Makerere/beans` di Hugging Face.
> Tidak perlu akun / API key.

In [ ]:
from datasets import load_dataset

CLASS_NAMES = ['angular_leaf_spot', 'bean_rust', 'healthy']
RAW_DIR     = 'beans_raw'

print('⏳ Mendownload dataset Beans dari Hugging Face...')
hf_dataset = load_dataset('AI-Lab-Makerere/beans', trust_remote_code=True)

print('\n📋 Info Dataset:')
print(f'   Nama   : Beans Disease Dataset')
print(f'   Kelas  : {CLASS_NAMES}')
for split_name, split_ds in hf_dataset.items():
    print(f'   Split {split_name:<12}: {len(split_ds):,} gambar')
total_raw = sum(len(s) for s in hf_dataset.values())
print(f'   TOTAL  : {total_raw:,} gambar')

## 2. Ekspor Dataset ke Folder

In [ ]:
print('⏳ Mengekspor gambar ke folder...')
exported = {cls: 0 for cls in CLASS_NAMES}

# Deteksi key label
sample_row = next(iter(hf_dataset[list(hf_dataset.keys())[0]]))
label_key  = 'label' if 'label' in sample_row else 'labels'
print(f'   Key label: "{label_key}"')

for split_name, split_ds in hf_dataset.items():
    for row in split_ds:
        img   = row['image']
        label = row[label_key]
        cls   = CLASS_NAMES[label]
        dest  = os.path.join(RAW_DIR, cls)
        os.makedirs(dest, exist_ok=True)
        fname = f'{cls}_{exported[cls]:04d}.jpg'
        img.convert('RGB').save(os.path.join(dest, fname), 'JPEG', quality=95)
        exported[cls] += 1

print('\n✅ Export selesai!')
for cls, cnt in exported.items():
    print(f'   {cls:<25}: {cnt:,} gambar')
print(f'   {"TOTAL":<25}: {sum(exported.values()):,} gambar')

## 3. Eksplorasi Dataset

In [ ]:
# Cek distribusi & bukti resolusi tidak seragam
print('📊 Distribusi gambar:')
sizes = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(RAW_DIR, cls)
    files = [f for f in os.listdir(cls_path) if f.endswith('.jpg')]
    print(f'   {cls:<25}: {len(files):,}')
    for f in random.sample(files, min(5, len(files))):
        with Image.open(os.path.join(cls_path, f)) as im:
            sizes.append(im.size)

print('\n📐 Sampel resolusi gambar (tidak seragam):')
for s in list(set(sizes))[:8]:
    print(f'   {s[0]} x {s[1]} px')

In [ ]:
# Visualisasi sampel
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Contoh Gambar per Kelas — Beans Disease Dataset', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(RAW_DIR, cls)
    files = [f for f in os.listdir(cls_path) if f.endswith('.jpg')]
    for col, fname in enumerate(random.sample(files, 4)):
        img = mpimg.imread(os.path.join(cls_path, fname))
        axes[row][col].imshow(img)
        axes[row][col].set_title(cls.replace('_',' '), fontsize=9)
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Pembagian Dataset — Train / Validation / Test (70 / 15 / 15)

In [ ]:
BASE_SPLIT_DIR = 'beans_split'
random.seed(42)
split_counts = {'train':{}, 'validation':{}, 'test':{}}

for cls in CLASS_NAMES:
    src_path   = os.path.join(RAW_DIR, cls)
    all_images = [f for f in os.listdir(src_path) if f.endswith('.jpg')]
    random.shuffle(all_images)
    n       = len(all_images)
    n_train = math.floor(n * 0.70)
    n_val   = math.floor(n * 0.15)
    splits  = {
        'train':      all_images[:n_train],
        'validation': all_images[n_train:n_train+n_val],
        'test':       all_images[n_train+n_val:]
    }
    for split_name, imgs in splits.items():
        dest_dir = os.path.join(BASE_SPLIT_DIR, split_name, cls)
        os.makedirs(dest_dir, exist_ok=True)
        for f in imgs:
            shutil.copy(os.path.join(src_path, f), dest_dir)
        split_counts[split_name][cls] = len(imgs)

print(f'{"Kelas":<25} {"Train":>7} {"Val":>7} {"Test":>7}')
print('-' * 48)
for cls in CLASS_NAMES:
    print(f'{cls:<25} {split_counts["train"][cls]:>7} {split_counts["validation"][cls]:>7} {split_counts["test"][cls]:>7}')
print('-' * 48)
tr = sum(split_counts['train'].values())
vl = sum(split_counts['validation'].values())
te = sum(split_counts['test'].values())
print(f'{"TOTAL":<25} {tr:>7} {vl:>7} {te:>7}')
print(f'\n✅ Total: {tr+vl+te:,} gambar')

## 5. ImageDataGenerator

> Resolusi gambar asli tidak seragam — resize ke 224×224 hanya dilakukan saat loading via `target_size`,
> bukan preprocessing di disk.

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(BASE_SPLIT_DIR, 'train'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=42
)
val_generator = val_test_datagen.flow_from_directory(
    os.path.join(BASE_SPLIT_DIR, 'validation'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    os.path.join(BASE_SPLIT_DIR, 'test'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Class indices : {train_generator.class_indices}')
print(f'Train batches : {len(train_generator)}')
print(f'Val batches   : {len(val_generator)}')
print(f'Test batches  : {len(test_generator)}')

## 6. Arsitektur Model CNN — Sequential + Conv2D + MaxPooling

In [ ]:
NUM_CLASSES = len(CLASS_NAMES)  # 3

model = Sequential([
    # Blok 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same',
                  input_shape=(224, 224, 3), name='conv2d_1'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu', padding='same', name='conv2d_1b'),
    layers.MaxPooling2D(2, 2, name='maxpool_1'),
    layers.Dropout(0.25),

    # Blok 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same', name='conv2d_2'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same', name='conv2d_2b'),
    layers.MaxPooling2D(2, 2, name='maxpool_2'),
    layers.Dropout(0.25),

    # Blok 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same', name='conv2d_3'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), activation='relu', padding='same', name='conv2d_3b'),
    layers.MaxPooling2D(2, 2, name='maxpool_3'),
    layers.Dropout(0.25),

    # Blok 4
    layers.Conv2D(256, (3,3), activation='relu', padding='same', name='conv2d_4'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2, name='maxpool_4'),
    layers.Dropout(0.25),

    # Fully Connected
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),

    # Output
    layers.Dense(NUM_CLASSES, activation='softmax', name='output')
], name='beans_cnn')

model.summary()

## 7. Compile & Training dengan Callbacks

- **EarlyStopping** — berhenti jika `val_accuracy` tidak naik selama 15 epoch
- **ReduceLROnPlateau** — kurangi learning rate jika `val_loss` stagnan 5 epoch
- **ModelCheckpoint** — simpan bobot terbaik ke `best_model.keras`

In [ ]:
import json as _json

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print('🚀 Memulai training...')
history = model.fit(
    train_generator,
    epochs=60,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ Training selesai!')

# Simpan history ke file — aman dari runtime restart
with open('training_history.json', 'w') as _f:
    _json.dump(history.history, _f)
print('✅ History tersimpan: training_history.json')

## 8. Plot Akurasi & Loss

In [ ]:
import json as _json

# Load history dari file jika variabel 'history' tidak ada di memori
# (terjadi saat runtime restart setelah training selesai)
if 'history' not in dir() or not hasattr(history, 'history'):
    print('⚠️  Variabel history tidak ditemukan — memuat dari training_history.json...')
    with open('training_history.json', 'r') as _f:
        _hist_data = _json.load(_f)
    class _HistWrapper:
        def __init__(self, d): self.history = d
    history = _HistWrapper(_hist_data)
    print('✅ History berhasil dimuat!')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Performa Model CNN — Beans Disease Classification', fontsize=14, fontweight='bold')
ep = range(1, len(history.history['accuracy']) + 1)

ax1.plot(ep, history.history['accuracy'],     'b-o', ms=3, lw=2, label='Train Accuracy')
ax1.plot(ep, history.history['val_accuracy'], 'r-o', ms=3, lw=2, label='Validation Accuracy')
ax1.axhline(y=0.85, color='green', linestyle='--', alpha=0.7, label='Target 85%')
ax1.set_title('Akurasi Model'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Akurasi')
ax1.legend(); ax1.set_ylim([0, 1.05]); ax1.grid(True, alpha=0.3)

ax2.plot(ep, history.history['loss'],     'b-o', ms=3, lw=2, label='Train Loss')
ax2.plot(ep, history.history['val_loss'], 'r-o', ms=3, lw=2, label='Validation Loss')
ax2.set_title('Loss Model'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot tersimpan: training_plot.png')

## 9. Evaluasi Model

In [ ]:
model = keras.models.load_model('best_model.keras')

train_generator.reset()
train_loss, train_acc = model.evaluate(train_generator, verbose=0)
test_loss,  test_acc  = model.evaluate(test_generator,  verbose=0)

print('=' * 48)
print('         HASIL EVALUASI MODEL')
print('=' * 48)
print(f'  Train Accuracy : {train_acc*100:.2f}%  {"✅" if train_acc >= 0.85 else "⚠️  (target 85%)"}')
print(f'  Train Loss     : {train_loss:.4f}')
print(f'  Test Accuracy  : {test_acc*100:.2f}%  {"✅" if test_acc >= 0.85 else "⚠️  (target 85%)"}')
print(f'  Test Loss      : {test_loss:.4f}')
print('=' * 48)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_generator.reset()
y_pred = np.argmax(model.predict(test_generator, verbose=0), axis=1)
y_true = test_generator.classes

print('📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=[c.replace('_','\n') for c in CLASS_NAMES],
            yticklabels=[c.replace('_','\n') for c in CLASS_NAMES])
plt.title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.ylabel('Label Sebenarnya'); plt.xlabel('Label Prediksi')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Simpan Model — SavedModel, TF-Lite, TFJS

> **Catatan Keras 3:** Gunakan `model.export()` untuk SavedModel (bukan `model.save()`).
> `model.save()` di Keras 3 hanya menerima `.keras` atau `.h5`.

In [ ]:
# ── FORMAT 1: SavedModel ──────────────────────────────────────────────────
# Keras 3: wajib pakai model.export() untuk SavedModel format
SAVED_MODEL_PATH = 'saved_model'
model.export(SAVED_MODEL_PATH)

print(f'✅ SavedModel → {SAVED_MODEL_PATH}/')
for item in os.listdir(SAVED_MODEL_PATH):
    print(f'   - {item}')

In [ ]:
# ── FORMAT 2: TF-Lite ─────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

os.makedirs('tflite', exist_ok=True)
TFLITE_PATH = 'tflite/model.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

with open('tflite/label.txt', 'w') as f:
    f.write('\n'.join(CLASS_NAMES))

print(f'✅ TF-Lite → {TFLITE_PATH}  ({os.path.getsize(TFLITE_PATH)/1e6:.2f} MB)')

In [ ]:
# Verifikasi TF-Lite
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()
print(f'✅ TF-Lite OK')
print(f'   Input  : {inp[0]["shape"]}  dtype={inp[0]["dtype"]}')
print(f'   Output : {out[0]["shape"]}')

In [ ]:
# ── FORMAT 3: TensorFlow.js ───────────────────────────────────────────────
TFJS_DIR = 'tfjs_model'
result = subprocess.run(
    ['tensorflowjs_converter', '--input_format=tf_saved_model',
     '--output_format=tfjs_graph_model', SAVED_MODEL_PATH, TFJS_DIR],
    capture_output=True, text=True
)
if result.returncode == 0:
    files_tfjs = os.listdir(TFJS_DIR)
    print(f'✅ TFJS → {TFJS_DIR}/  ({len(files_tfjs)} file)')
    for f in sorted(files_tfjs):
        print(f'   - {f}  ({os.path.getsize(os.path.join(TFJS_DIR, f))/1024:.1f} KB)')
else:
    print(f'❌ TFJS error: {result.stderr[:400]}')

## 11. Inference dengan TF-Lite

Bukti inferensi menggunakan model TF-Lite pada gambar dari test set (1 gambar per kelas).

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp_det = interpreter.get_input_details()
out_det = interpreter.get_output_details()

# Ambil 1 gambar per kelas dari test set
test_samples = []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(BASE_SPLIT_DIR, 'test', cls)
    files   = [f for f in os.listdir(cls_dir) if f.endswith('.jpg')]
    test_samples.append((cls, os.path.join(cls_dir, random.choice(files))))

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('🔍 Hasil Inference TF-Lite — Beans Disease Classification',
             fontsize=13, fontweight='bold')

print('=' * 62)
print(f'{"No":<4} {"Label Asli":<25} {"Prediksi":<25} {"Confidence":>10}')
print('=' * 62)

for i, (true_cls, img_path) in enumerate(test_samples):
    img_pil   = Image.open(img_path).convert('RGB').resize((224, 224))
    img_array = np.array(img_pil, dtype=np.float32) / 255.0
    img_input = np.expand_dims(img_array, axis=0)

    interpreter.set_tensor(inp_det[0]['index'], img_input)
    interpreter.invoke()
    output     = interpreter.get_tensor(out_det[0]['index'])[0]
    pred_idx   = int(np.argmax(output))
    pred_cls   = CLASS_NAMES[pred_idx]
    confidence = float(np.max(output)) * 100
    status     = '✅' if pred_cls == true_cls else '❌'

    print(f'{i+1:<4} {true_cls:<25} {pred_cls:<25} {confidence:>9.2f}%  {status}')

    color = 'green' if pred_cls == true_cls else 'red'
    axes[i].imshow(img_pil)
    axes[i].set_title(
        f'Asli    : {true_cls}\nPrediksi: {pred_cls}\nConf    : {confidence:.1f}%',
        fontsize=9, color=color
    )
    axes[i].axis('off')

print('=' * 62)
plt.tight_layout()
plt.savefig('inference_result.png', dpi=130, bbox_inches='tight')
plt.show()
print('\n✅ Inference selesai! Tersimpan: inference_result.png')

## 12. Package & Download

In [ ]:
def zip_folder(src, dst):
    with zipfile.ZipFile(dst, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(src):
            for file in files:
                zf.write(os.path.join(root, file))
    print(f'  📦 {dst}  ({os.path.getsize(dst)/1e6:.2f} MB)')

zip_folder('saved_model', 'saved_model.zip')
zip_folder('tfjs_model',  'tfjs_model.zip')

from google.colab import files
for fname in ['saved_model.zip', 'tflite/model.tflite', 'tflite/label.txt',
              'tfjs_model.zip', 'training_plot.png', 'confusion_matrix.png',
              'inference_result.png']:
    files.download(fname)
    print(f'  ⬇️  {fname}')
print('✅ Semua file didownload!')

## 13. Ringkasan Akhir

In [ ]:
best_val = max(history.history['val_accuracy'])
n_epochs = len(history.history['accuracy'])

print('=' * 52)
print('           RINGKASAN PROYEK CNN')
print('=' * 52)
print(f'  Dataset         : Beans Disease (HF)')
print(f'  Total Gambar    : {tr+vl+te:,}')
print(f'  Jumlah Kelas    : {NUM_CLASSES} (angular_leaf_spot, bean_rust, healthy)')
print(f'  Resolusi        : Tidak seragam (bervariasi)')
print(f'  Epochs Dilatih  : {n_epochs}')
print(f'  Best Val Acc    : {best_val*100:.2f}%')
print(f'  Train Accuracy  : {train_acc*100:.2f}%  {"✅" if train_acc>=0.85 else "⚠️"}')
print(f'  Test Accuracy   : {test_acc*100:.2f}%  {"✅" if test_acc>=0.85 else "⚠️"}')
print('=' * 52)
print('  Callbacks : EarlyStopping ✅  ReduceLR ✅  Checkpoint ✅')
print('  SavedModel ✅   TF-Lite ✅   TFJS ✅')
print('  Inference (TF-Lite) ✅')
print('=' * 52)